In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(VGAM)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(piano)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(foreach)
    library(DelayedArray)
})

In [ ]:
cds_VCS_raw = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered.RDS")

# Compute scores

In [ ]:
subsample_cds = function(cds) {
    metadata = as.data.frame(colData(cds))
    count_df = metadata %>% count(Experiment, Embryo, Marker, Phenotype, Bin)

    cells_sampled = c()

    for (i in 1:nrow(count_df)) {
        marker = count_df[i,]$Marker
        phenotype = count_df[i,]$Phenotype
        bin = count_df[i,]$Bin
        experiment = count_df[i,]$Experiment
        embryo = count_df[i,]$Embryo

        n = min(count_df[count_df$Marker == marker & count_df$Phenotype == phenotype & 
                         count_df$Experiment == experiment & count_df$Embryo == embryo,]$n)

        cells = metadata %>% filter(Marker == marker & Phenotype == phenotype & Bin == bin & 
                                    Experiment == experiment & Embryo == embryo) %>% pull(Cell)
        cells_sampled = c(cells_sampled, sample(cells, size=n, replace=F))
    }

    cds_subsampled = cds[, cds$Cell %in% cells_sampled]
    
    return(cds_subsampled)
}

compute_bin_score = function(cds, downsample = TRUE) {
    if (downsample) {
        cds_subsampled = subsample_cds(cds)
    } else {
        cds_subsampled = cds
    }
    
    metadata = as.data.frame(colData(cds_subsampled))

    bin_score_list_cluster = list()

    for (marker in unique(metadata$Marker)) {
        bin_score_list_cluster[[marker]] = list()
        for (phenotype in unique(metadata[metadata$Marker == marker,]$Phenotype)) {
            
            median_df = metadata %>%
                filter(Marker == marker & Phenotype == phenotype) %>%
                group_by(Experiment, Embryo, trajectory_refined, Cluster_sub, Bin) %>%
                summarise(med_median = median(Bin_Median)) %>%
                as.data.frame() %>%
                tidyr::spread(Bin, med_median, fill = 0) %>%
                dplyr::rename(Q1_median = Q1, Q2_median = Q2, Q3_median = Q3, Q4_median = Q4)
            
            bin_scores = metadata %>%
                filter(Marker == marker & Phenotype == phenotype) %>%
                group_by(Experiment, Embryo, Bin, trajectory_refined, Cluster_sub) %>%
                count() %>%
                as.data.frame() %>%
                tidyr::spread(Bin, n, fill = 0) %>%
                mutate(n = Q1+Q2+Q3+Q4) %>%
                as.data.frame() %>%
                filter(n >= 30) %>%
                left_join(., median_df, by = c("Experiment", "Embryo", "trajectory_refined", "Cluster_sub")) %>%
                rowwise() %>%
                mutate(score = (Q1_median*Q1 + Q2_median*Q2 + Q3_median*Q3 + Q4_median*Q4) / n) %>%
                as.data.frame()

            bin_scores = bin_scores %>%
                            mutate(Marker = marker,
                                   Phenotype = phenotype)

            bin_score_list_cluster[[marker]][[phenotype]] = bin_scores
        }
    }
    
    bin_score_concat = NULL
    for (marker in names(bin_score_list_cluster)) {
        df = do.call("rbind", bin_score_list_cluster[[marker]])
        if (is.null(bin_score_concat)) {
            bin_score_concat = df
        } else {
            bin_score_concat = rbind(bin_score_concat, df)
        }
    }

    metadata = as.data.frame(colData(cds))
    metadata = left_join(metadata,
          bin_score_concat %>%
              select(-c(Q1, Q2, Q3, Q4, Q1_median, Q2_median, Q3_median, Q4_median, n)), 
          by = c("Experiment", "Embryo", "trajectory_refined", "Cluster_sub", "Marker", "Phenotype")) %>%
    na.omit()
    
    cds = cds[, cds$Cell %in% metadata$Cell]
    cds$Score = metadata[match(cds$Cell, metadata$Cell), ]$score
    cds$Timepoint = as.factor(sapply(cds$Embryo, function(x) strsplit(x, '-')[[1]][1]))
    
    num_cells_expressed_cutoff = round(nrow(as.data.frame(colData(cds)))*0.01)

    cds = detect_genes(cds)
    cds = cds[rowData(cds)$num_cells_expressed > num_cells_expressed_cutoff, ]
    
    return(cds)
}

scale_norm = function(score) {
    (score - min(score))/(max(score)-min(score))
}

get_average_bin_scores = function(cds, downsample = TRUE, n = 30) {
    scores = NULL
    for (i in 1:n) {
        print(i)
        flush.console()
        cds_scores = compute_bin_score(cds, downsample)
        score_df = as.data.frame(colData(cds_scores)) %>%
                        distinct(Experiment, Embryo, Marker, Phenotype, trajectory_refined, Cluster_sub, Score, .keep_all=T) %>%
                        mutate(i = i)
        scores = if (is.null(scores)) {
            scores = score_df
        } else {
            scores = rbind(scores, score_df)
        } 
    }

    scores = scores %>%
        group_by(Experiment, Embryo, Marker, Phenotype, trajectory_refined, Cluster_sub) %>%
        mutate(avg_score = mean(Score), med_score = median(Score), sd_score = sd(Score)) %>%
        as.data.frame() %>%
        mutate(sd_score = ifelse(is.na(sd_score), 0, sd_score)) %>%
        distinct(Experiment, Embryo, Marker, Phenotype, trajectory_refined, Cluster_sub, .keep_all=T) %>%
        select(Experiment, Embryo, Marker, Phenotype, trajectory_refined, Cluster_sub, avg_score, med_score, sd_score, -i)

    metadata = as.data.frame(colData(cds))
    
    metadata = metadata %>%
        dplyr::select(-dplyr::any_of(c("avg_score", "med_score", "sd_score", "scaled_score", "scaled_score_trajectory", "Score")))
        
    metadata = left_join(metadata, scores, 
          by = c("Experiment", "Embryo", "trajectory_refined", "Cluster_sub", "Marker", "Phenotype")) %>%
        na.omit()
    
    scaled_scores = metadata %>%
                        distinct(trajectory_refined, Cluster_sub, Marker, Phenotype, .keep_all=T) %>%
                        group_by(Marker, Phenotype) %>%
                        mutate(scaled_score = scale_norm(avg_score)) %>%
                        select(trajectory_refined, Cluster_sub, Marker, Phenotype, scaled_score) %>%
                        as.data.frame()

    metadata = metadata %>%
        left_join(., scaled_scores, by = c("trajectory_refined", "Cluster_sub", "Marker", "Phenotype"))
    
    scaled_scores = metadata %>%
                        distinct(trajectory_refined, Cluster_sub, Marker, Phenotype, .keep_all=T) %>%
                        group_by(trajectory_refined, Marker, Phenotype) %>%
                        mutate(scaled_score_trajectory = scale_norm(avg_score)) %>%
                        select(trajectory_refined, Cluster_sub, Marker, Phenotype, scaled_score_trajectory) %>%
                        as.data.frame()
    
    metadata = metadata %>%
        left_join(., scaled_scores, by = c("trajectory_refined", "Cluster_sub", "Marker", "Phenotype")) %>%
        mutate(scaled_score_trajectory = ifelse(is.na(scaled_score_trajectory), 
                                                scaled_score, scaled_score_trajectory))

    
    cds = cds[, cds$Cell %in% metadata$Cell]
    cds$avg_score = metadata[match(cds$Cell, metadata$Cell), ]$avg_score
    cds$med_score = metadata[match(cds$Cell, metadata$Cell), ]$med_score
    cds$sd_score = metadata[match(cds$Cell, metadata$Cell), ]$sd_score
    cds$scaled_score = metadata[match(cds$Cell, metadata$Cell), ]$scaled_score
    cds$scaled_score_trajectory = metadata[match(cds$Cell, metadata$Cell), ]$scaled_score_trajectory
    cds$Timepoint = as.factor(sapply(cds$Embryo, function(x) strsplit(x, '-')[[1]][1]))
    
    num_cells_expressed_cutoff = round(nrow(as.data.frame(colData(cds)))*0.01)

    cds = detect_genes(cds)
    cds = cds[rowData(cds)$num_cells_expressed > num_cells_expressed_cutoff, ]
                                         
    return(cds)
}

In [ ]:
cds_VCS_embryo = get_average_bin_scores(cds_VCS_raw, downsample = T, n = 30)

In [ ]:
saveRDS(cds_VCS_embryo, 
        "../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30.RDS")